# Module 1 API — Steps 0 & 1 (cleaned up, package-ized)

This notebook replaces `api-ready-version (1).ipynb`. It applies:

- **Step 0** — removes the broken shuttle-tracking stub from `/api/process-video`
  (the `TEMP_DIR` reassignment bug that raised `UnboundLocalError` on every request).
- **Step 1** — moves off the flat `temp_exports/` folder onto the shared
  `BASE / uploads / outputs / models / static` layout, and standardizes
  `match_id` / `player_id` validation to match Module 2's stricter
  `p001` / `p001_m0001` format (`validate_player_match_metadata`, now
  shared via `app/config.py`).

Everything else — the tracker, the movement-feature math, the rendering —
is byte-for-byte the same logic as the original notebook, just reorganized
into the package layout from the roadmap's Section 1.

Run top to bottom on Kaggle with GPU + Internet enabled.


In [ ]:
!pip install fastapi uvicorn pyngrok nest-asyncio ultralytics python-multipart -q

In [ ]:
import os
from pathlib import Path

# Point the whole app at Kaggle's writable working directory.
os.environ["BADMINTON_APP_BASE"] = "/kaggle/working"
BASE = Path(os.environ["BADMINTON_APP_BASE"])

for sub in ["app", "app/module1", "app/module2", "app/routers", "app/shared", "static"]:
    (BASE / sub).mkdir(parents=True, exist_ok=True)

print("Writing package files under:", BASE)

## Package files (`%%writefile`, same pattern Module 2's notebook already uses)

In [ ]:
%%writefile app/__init__.py


In [ ]:
%%writefile app/config.py
"""
Shared configuration for the badminton analysis API.

Both Module 1 (pose/movement) and Module 2 (shuttle/tactical) read their
paths, model locations, and identifier rules from here so the two pipelines
can never disagree about where uploads/outputs live or what a valid
match_id/player_id looks like.

Folder structure expected on disk (all relative to BASE):

    project/
      app/                   <- this package
      TrackNetV3/            cloned + patched (Step 2)
      models/
        best.pt               Module 1 shot/stance classifier
        player_best.pt        Module 2 player detector + shot classifier
        court_best.pt         Module 2 court keypoint detector
      uploads/
      outputs/
        combine/              Module 2 "p1" tactical output (Step 11)
        m2/                   Module 2 "p2" per-shot landing output (Step 11)
      static/

Override the root with the BADMINTON_APP_BASE env var, e.g. "/kaggle/working".
"""
import os
import re
from pathlib import Path

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
BASE = Path(os.getenv("BADMINTON_APP_BASE", Path(__file__).resolve().parent.parent))

TRACKNET_DIR = BASE / "TrackNetV3"
MODELS_DIR = BASE / "models"
UPLOADS_DIR = BASE / "uploads"
OUTPUTS_DIR = BASE / "outputs"
STATIC_DIR = BASE / "static"

# Module 2 writes its two JSON deliverables into these (Step 11)
COMBINE_OUTPUTS_DIR = OUTPUTS_DIR / "combine"
M2_OUTPUTS_DIR = OUTPUTS_DIR / "m2"

for _dir in (UPLOADS_DIR, OUTPUTS_DIR, COMBINE_OUTPUTS_DIR, M2_OUTPUTS_DIR, STATIC_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Model paths
# ------------------------------------------------------------------
# Module 1 — generic pose model + Hirusha's fine-tuned 7-class shot/stance
# classifier. Same defaults as the original notebook; override via env var
# instead of editing this file.
POSE_MODEL_PATH = os.getenv("POSE_MODEL_PATH", "yolov8n-pose.pt")
MODULE1_CLASSIFIER_MODEL_PATH = Path(os.getenv(
    "MODULE1_CLASSIFIER_MODEL_PATH",
    "/kaggle/input/models/chathushikavindya/best-pt/pytorch/default/1/best.pt",
))

# Module 2 (Steps 4-5)
PLAYER_MODEL_PATH = Path(os.getenv("PLAYER_MODEL_PATH", MODELS_DIR / "player_best.pt"))
COURT_MODEL_PATH = Path(os.getenv("COURT_MODEL_PATH", MODELS_DIR / "court_best.pt"))

# TrackNetV3 checkpoints (Step 2)
TRACKNET_PT = Path(os.getenv("TRACKNET_PT", TRACKNET_DIR / "ckpts" / "TrackNet_best.pt"))
INPAINT_PT = Path(os.getenv("INPAINT_PT", TRACKNET_DIR / "ckpts" / "InpaintNet_best.pt"))


# ------------------------------------------------------------------
# Shared match_id / player_id contract (roadmap finding 0.5 / Step 1)
#
# Module 1 originally accepted free-form strings ("match_001", "player_01").
# Module 2 already validates strictly. We standardize on Module 2's format
# since it maps directly onto the Firestore schema shape
# players/{player_id}/matches/{match_id}. Copied verbatim from Module 2's
# pipeline (pipeline_main.py / cell 18, line ~2384) — logic unchanged, only
# relocated so both endpoints import the same function instead of each
# enforcing their own rule.
# ------------------------------------------------------------------
def validate_player_match_metadata(player_id, player_name, match_id):
    player_id = str(player_id).strip()
    player_name = str(player_name).strip()
    match_id = str(match_id).strip()

    if not re.fullmatch(r"p\d{3}", player_id):
        raise ValueError(
            "player_id must use the format p001, p002, etc."
        )

    expected_pattern = rf"{re.escape(player_id)}_m\d{{4}}"

    if not re.fullmatch(expected_pattern, match_id):
        raise ValueError(
            f"match_id must use the format {player_id}_m0001."
        )

    match_number = int(match_id.rsplit("_m", 1)[1])

    if not 1 <= match_number <= 20:
        raise ValueError(
            "Match number must be between 0001 and 0020."
        )

    if not player_name:
        raise ValueError("player_name cannot be empty.")

    return player_id, player_name, match_id


In [ ]:
%%writefile app/shared/__init__.py


In [ ]:
%%writefile app/shared/video_utils.py
"""Shared video transcoding helper.

OpenCV's VideoWriter produces mp4v-codec files that most browsers can't
play back directly. Module 1's notebook already had a plain ffmpeg
subprocess call for this (cell 3); Module 2's notebook used a second,
independent imageio_ffmpeg-based path for the same job. We keep only this
one so the whole system depends on a single ffmpeg installation (roadmap
finding 0.6 / Step 12).
"""
import subprocess


def transcode_to_h264(input_path, output_path):
    subprocess.run([
        "ffmpeg", "-y",
        "-i", str(input_path),
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-preset", "fast",
        str(output_path),
    ], check=True)


In [ ]:
%%writefile app/module1/__init__.py


In [ ]:
%%writefile app/module1/tracking.py
"""
Module 1 — player detection & locked-target tracking.

This is Hirusha's own IoU/histogram tracker, independent of Module 2's
DeepSort-based tracker (roadmap finding 0.2 — the two are separate tracking
approaches and neither replaces the other). Content moved verbatim out of
the original api-ready notebook's cells 2 (CONFIG), 4 (tracker helpers),
and 6 (run_tracking_inference).
"""
import csv
import json

import cv2
import numpy as np
from ultralytics import YOLO

# ------------------------------------------------------------------
# CONFIG (notebook cell 2)
# ------------------------------------------------------------------
CONF_THRES = 0.35
KEYPOINT_CONF_THRES = 0.25

MAX_MISSING_FRAMES = 40
MAX_CENTER_DISTANCE = 200
MIN_MATCH_SCORE = 0.25

COURT_Y_MIN = 0.28
COURT_Y_MAX = 0.62

MOTION_THRESHOLD = 0.04

KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

SKELETON = [
    (5, 7), (7, 9),
    (6, 8), (8, 10),
    (5, 6),
    (5, 11), (6, 12),
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (0, 1), (0, 2),
    (1, 3), (2, 4)
]


# ------------------------------------------------------------------
# Locked-target tracker helpers (notebook cell 4)
# ------------------------------------------------------------------
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]);  yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]);  yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])
    return inter / (areaA + areaB - inter + 1e-6)


def bbox_center(box):
    x1, y1, x2, y2 = box
    return np.array([(x1 + x2) / 2, (y1 + y2) / 2])


def center_distance(boxA, boxB):
    return np.linalg.norm(bbox_center(boxA) - bbox_center(boxB))


def bbox_area(box):
    x1, y1, x2, y2 = box
    return max(0, x2 - x1) * max(0, y2 - y1)


def safe_crop(frame, box):
    h, w = frame.shape[:2]
    x1 = int(max(0, min(w - 1, box[0])))
    y1 = int(max(0, min(h - 1, box[1])))
    x2 = int(max(0, min(w - 1, box[2])))
    y2 = int(max(0, min(h - 1, box[3])))
    if x2 <= x1 or y2 <= y1:
        return None
    return frame[y1:y2, x1:x2]


def color_histogram(frame, box):
    crop = safe_crop(frame, box)
    if crop is None or crop.size == 0:
        return None
    crop = cv2.resize(crop, (64, 128))
    hsv  = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [32, 32], [0, 180, 0, 256])
    cv2.normalize(hist, hist)
    return hist


def histogram_similarity(histA, histB):
    if histA is None or histB is None:
        return 0.0
    return max(0.0, min(1.0, cv2.compareHist(histA, histB, cv2.HISTCMP_CORREL)))


def motion_score(frame, prev_frame, box):
    if prev_frame is None:
        return 1.0
    crop_curr = safe_crop(frame,      box)
    crop_prev = safe_crop(prev_frame, box)
    if crop_curr is None or crop_prev is None:
        return 0.0
    if crop_curr.shape != crop_prev.shape:
        crop_prev = cv2.resize(crop_prev, (crop_curr.shape[1], crop_curr.shape[0]))
    diff  = cv2.absdiff(crop_curr, crop_prev)
    gray  = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    moved = np.count_nonzero(gray > 25)
    return moved / (gray.size + 1e-6)


def is_inside_court(box, frame_h):
    foot_y = box[3] / frame_h
    return COURT_Y_MIN < foot_y < COURT_Y_MAX


def face_visibility_score(kpt_conf):
    score  = sum(2 for i in [0, 1, 2, 3, 4] if kpt_conf[i] > KEYPOINT_CONF_THRES)
    score += sum(1 for i in [5, 6]           if kpt_conf[i] > KEYPOINT_CONF_THRES)
    return score


def extract_detections(result, frame):
    detections = []
    if result.boxes is None or result.keypoints is None:
        return detections
    boxes     = result.boxes.xyxy.cpu().numpy()
    det_confs = result.boxes.conf.cpu().numpy()
    kpts_xy   = result.keypoints.xy.cpu().numpy()
    kpts_conf = result.keypoints.conf.cpu().numpy() \
                if result.keypoints.conf is not None \
                else np.ones((len(boxes), 17))
    for i in range(len(boxes)):
        if det_confs[i] < CONF_THRES:
            continue
        detections.append({
            "box":           boxes[i].tolist(),
            "det_conf":      float(det_confs[i]),
            "keypoints":     kpts_xy[i].tolist(),
            "keypoint_conf": kpts_conf[i].tolist(),
            "hist":          None,
        })
    return detections


def select_initial_target(detections, frame, prev_frame, frame_h):
    best_det, best_score = None, -1

    for det in detections:
        box      = det["box"]
        kpt_conf = np.array(det["keypoint_conf"])

        if not is_inside_court(box, frame_h):
            continue

        mv = motion_score(frame, prev_frame, box)
        if mv < MOTION_THRESHOLD:
            continue

        if det.get("hist") is None:
            det["hist"] = color_histogram(frame, box)

        score = (
            face_visibility_score(kpt_conf)                  * 4.0 +
            det["det_conf"]                                  * 2.0 +
            min(bbox_area(box) / (frame_h * frame_h), 1.0)  * 1.0 +
            min(mv / 0.20, 1.0)                              * 1.0
        )
        if score > best_score:
            best_score, best_det = score, det

    return best_det


def match_locked_target(detections, frame, prev_frame, locked_box, locked_hist, frame_h):
    best_det, best_score = None, -1

    for det in detections:
        box = det["box"]

        if not is_inside_court(box, frame_h):
            continue

        iou  = compute_iou(locked_box, box)
        dist = center_distance(locked_box, box)
        if dist > MAX_CENTER_DISTANCE and iou < 0.05:
            continue

        area_ratio  = bbox_area(box) / (bbox_area(locked_box) + 1e-6)
        mv          = motion_score(frame, prev_frame, box)

        if det.get("hist") is None:
            det["hist"] = color_histogram(frame, box)

        match_score = (
            iou                                              * 4.0 +
            max(0.0, 1.0 - dist / MAX_CENTER_DISTANCE)      * 2.0 +
            histogram_similarity(locked_hist, det["hist"])   * 3.0 +
            det["det_conf"]                                  * 1.0 +
            (1.0 - min(abs(1.0 - area_ratio), 1.0))         * 1.0
        ) / 11.0

        if match_score > best_score:
            best_score, best_det = match_score, det

    return (None, best_score) if best_score < MIN_MATCH_SCORE else (best_det, best_score)


def draw_pose(frame, kpts, kpt_conf):
    for p1, p2 in SKELETON:
        if kpt_conf[p1] > KEYPOINT_CONF_THRES and kpt_conf[p2] > KEYPOINT_CONF_THRES:
            cv2.line(frame,
                     (int(kpts[p1][0]), int(kpts[p1][1])),
                     (int(kpts[p2][0]), int(kpts[p2][1])),
                     (0, 255, 255), 2)
    for i, (x, y) in enumerate(kpts):
        if kpt_conf[i] > KEYPOINT_CONF_THRES:
            cv2.circle(frame, (int(x), int(y)), 4, (0, 0, 255), -1)


# ------------------------------------------------------------------
# Main inference entry point (notebook cell 6)
# ------------------------------------------------------------------
def run_tracking_inference(video_path, out_mp4, out_json, out_csv, model_path="yolov8n-pose.pt", custom_model_path="best.pt", progress_callback=None):
    baseline_model = YOLO(model_path)
    custom_model = YOLO(custom_model_path)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise Exception("Cannot open video")

    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        fps = 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    tracking_json = {
        "fps": fps,
        "frame_width": frame_w,
        "frame_height": frame_h,
        "frames": []
    }

    csv_rows = []

    locked = False
    locked_box = None
    locked_hist = None
    missing_count = 0
    last_good_detection = None
    prev_frame = None
    pose_name = "Detecting..."
    shot_conf = 0.0

    total_conf = 0.0
    detected_frames = 0

    for frame_id in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break

        timestamp = frame_id / fps

        base_result = baseline_model(frame, conf=CONF_THRES, verbose=False)[0]
        detections = extract_detections(base_result, frame)

        selected = None
        match_score = None

        if not locked:
            if len(detections) > 0:
                selected = select_initial_target(detections, frame, prev_frame, frame_h)
                if selected is not None:
                    locked = True
                    locked_box = selected["box"]
                    locked_hist = selected["hist"]
                    last_good_detection = selected
                    missing_count = 0
        else:
            selected, match_score = match_locked_target(detections, frame, prev_frame, locked_box, locked_hist, frame_h)
            if selected is not None:
                locked_box = selected["box"]
                new_hist = selected["hist"]
                if locked_hist is not None and new_hist is not None:
                    locked_hist = cv2.addWeighted(locked_hist, 0.85, new_hist, 0.15, 0)
                last_good_detection = selected
                missing_count = 0
            else:
                missing_count += 1
                if missing_count <= MAX_MISSING_FRAMES:
                    selected = last_good_detection
                else:
                    selected = None
                    locked = False
                    locked_box = None
                    locked_hist = None
                    last_good_detection = None
                    missing_count = 0

        frame_data = {
            "frame_id": frame_id,
            "timestamp": round(timestamp, 4),
            "player_detected": False,
            "tracking_status": "missing",
            "match_score": match_score,
            "shot_classification": pose_name,
            "shot_confidence": shot_conf,
            "bounding_box": None,
            "detection_confidence": None,
            "keypoints": []
        }

        if selected is not None:
            # Optimize: Only run the classification model every 5 frames if we have a tracked player
            if frame_id % 5 == 0:
                custom_result = custom_model(frame, verbose=False)[0]
                if custom_result.boxes and len(custom_result.boxes) > 0:
                    pose_id = int(custom_result.boxes.cls[0].item())
                    pose_name = custom_result.names[pose_id]
                    shot_conf = float(custom_result.boxes.conf[0].item())

            box = selected["box"]
            kpts = np.array(selected["keypoints"])
            kpt_conf = np.array(selected["keypoint_conf"])
            x1, y1, x2, y2 = map(int, box)

            if missing_count == 0:
                status = "tracked"
            else:
                status = "temporarily_predicted"

            frame_data["player_detected"] = True
            frame_data["tracking_status"] = status
            frame_data["bounding_box"] = {
                "x1": float(box[0]), "y1": float(box[1]), "x2": float(box[2]), "y2": float(box[3]),
                "width": float(box[2] - box[0]), "height": float(box[3] - box[1])
            }
            frame_data["detection_confidence"] = float(selected["det_conf"])

            total_conf += float(selected["det_conf"])
            detected_frames += 1

            for idx, name in enumerate(KEYPOINT_NAMES):
                score = float(kpt_conf[idx])
                kx, ky = float(kpts[idx][0]), float(kpts[idx][1])
                frame_data["keypoints"].append({
                    "id": idx, "name": name, "x": kx, "y": ky, "confidence": score
                })
                # Add to CSV row
                csv_rows.append([frame_id, 1, name, kx, ky, score])

        tracking_json["frames"].append(frame_data)
        prev_frame = frame.copy()

        if progress_callback:
            progress_callback(frame_id + 1, total_frames)

    cap.release()

    with open(out_json, "w") as f:
        json.dump(tracking_json, f, indent=4)

    with open(out_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["frame", "player_id", "keypoint", "x", "y", "confidence"])
        writer.writerows(csv_rows)

    avg_conf = (total_conf / max(detected_frames, 1)) * 100

    return {
        "player_tracked": detected_frames > 0,
        "keypoints_detected": 17,
        "average_confidence": round(avg_conf, 2)
    }


In [ ]:
%%writefile app/module1/movement.py
"""
Module 1 — movement feature extraction (speed, acceleration, jump height,
court zones, MEI, etc). Content moved verbatim out of the original
api-ready notebook's cells 2 (CONF_THR), 5 (small classification helpers),
and 8 (extract_movement_features).
"""
import json
import math

import numpy as np
import pandas as pd
from scipy.signal import find_peaks, savgol_filter

# Confidence threshold used when deciding whether a keypoint is reliable
# enough to feed into movement calculations (notebook cell 2).
CONF_THR = 0.25


# ------------------------------------------------------------------
# Small classification helpers (notebook cell 5)
# ------------------------------------------------------------------
def _movement_direction(dx_px, dy_px):
    """Angle in degrees. 0 = right, CCW positive."""
    if abs(dx_px) < 1e-3 and abs(dy_px) < 1e-3:
        return 0.0
    angle = math.degrees(math.atan2(-dy_px, dx_px))
    return round(angle % 360, 1)


def _classify_step(jump_px, speed, accel):
    if jump_px > 15:
        return 'jump'
    if speed > 4.0:
        return 'sprint'
    if abs(accel) > 5.0:
        return 'lunge'
    if speed > 1.0:
        return 'step'
    return 'stand'


# ------------------------------------------------------------------
# Main entry point (notebook cell 8)
# ------------------------------------------------------------------
def extract_movement_features(input_json_path, output_json_path, output_csv_path,
                               output_video_path=None, original_video_path=None,
                               match_id="match_001", player_id="player_01"):
    with open(input_json_path, 'r') as f:
        data = json.load(f)

    FPS        = data.get('fps', 30)
    FRAME_W    = data.get('frame_width', 1920)
    FRAME_H    = data.get('frame_height', 1080)
    frames_raw = data.get('frames', [])

    # Pixels → metres calibration (standard badminton court)
    COURT_M_HEIGHT = 13.4
    COURT_M_WIDTH  = 6.1
    PX_PER_M_Y = FRAME_H / COURT_M_HEIGHT
    PX_PER_M_X = FRAME_W / COURT_M_WIDTH

    # ── Build per-frame DataFrame (detected frames only) ──────────────────────
    rows = []
    for fr in frames_raw:
        if not fr.get('player_detected', False):
            continue

        bb  = fr['bounding_box']
        kps = {kp['name']: kp for kp in fr.get('keypoints', [])}

        def kp_xy(name):
            k = kps.get(name)
            if k and k['confidence'] >= CONF_THR:
                return k['x'], k['y']
            return np.nan, np.nan

        lax, lay = kp_xy('left_ankle')
        rax, ray = kp_xy('right_ankle')
        foot_x   = np.nanmean([lax, rax])
        foot_y   = np.nanmean([lay, ray])

        lhx, lhy = kp_xy('left_hip')
        rhx, rhy = kp_xy('right_hip')
        hip_x    = np.nanmean([lhx, rhx])
        hip_y    = np.nanmean([lhy, rhy])

        lwx, lwy = kp_xy('left_wrist')
        rwx, rwy = kp_xy('right_wrist')

        rows.append(dict(
            frame_id=fr['frame_id'],
            timestamp=fr['timestamp'],
            bb_x1=bb['x1'], bb_y1=bb['y1'], bb_x2=bb['x2'], bb_y2=bb['y2'],
            bb_w=bb['width'], bb_h=bb['height'],
            foot_x=foot_x, foot_y=foot_y,
            hip_x=hip_x,   hip_y=hip_y,
            lax=lax, lay=lay, rax=rax, ray=ray,
            lwx=lwx, lwy=lwy, rwx=rwx, rwy=rwy,
            lhx=lhx, lhy=lhy, rhx=rhx, rhy=rhy,
        ))

    df = pd.DataFrame(rows).sort_values('frame_id').reset_index(drop=True)
    if len(df) == 0:
        return {}

    # ── Smoothing window (≥3, odd, ~100 ms) ───────────────────────────────────
    WIN = max(3, int(FPS * 0.10))
    WIN = WIN if WIN % 2 == 1 else WIN + 1

    df['foot_x_sm'] = savgol_filter(df['foot_x'].ffill().bfill(), WIN, 2)
    df['foot_y_sm'] = savgol_filter(df['foot_y'].ffill().bfill(), WIN, 2)

    # ── Speed ──────────────────────────────────────────────────────────────────
    dx_px = df['foot_x_sm'].diff()
    dy_px = df['foot_y_sm'].diff()
    dt    = df['timestamp'].diff().replace(0, np.nan)

    dx_m = dx_px / PX_PER_M_X
    dy_m = dy_px / PX_PER_M_Y

    df['speed_mps'] = np.sqrt(dx_m**2 + dy_m**2) / dt
    df['speed_mps'] = df['speed_mps'].clip(upper=15)

    # ── Acceleration (store in df to share the same index as dt) ──────────────
    df['speed_sm']  = savgol_filter(df['speed_mps'].fillna(0), WIN, 2)
    df['accel_mps2'] = df['speed_sm'].diff() / dt
    df['accel_mps2'] = df['accel_mps2'].clip(-30, 30)

    # ── Court zone ────────────────────────────────────────────────────────────
    def lateral_zone(x):
        if pd.isna(x): return 'Unknown'
        frac = x / FRAME_W
        if frac < 0.33:   return 'Left'
        elif frac < 0.67: return 'Centre'
        else:             return 'Right'

    def depth_zone(y):
        if pd.isna(y): return 'Unknown'
        frac = y / FRAME_H
        if frac < 0.33:   return 'Net'
        elif frac < 0.67: return 'Mid'
        else:             return 'Back'

    df['lateral_zone'] = df['foot_x'].apply(lateral_zone)
    df['depth_zone']   = df['foot_y'].apply(depth_zone)
    df['court_zone']   = df['lateral_zone'] + '-' + df['depth_zone']

    # ── Stride frequency ───────────────────────────────────────────────────────
    left_y  = df['lay'].ffill().bfill().values
    right_y = df['ray'].ffill().bfill().values
    min_dist_frames = int(FPS * 0.15)

    left_peaks,  _ = find_peaks(-left_y,  distance=min_dist_frames, prominence=5)
    right_peaks, _ = find_peaks(-right_y, distance=min_dist_frames, prominence=5)
    total_peaks    = len(left_peaks) + len(right_peaks)
    total_seconds  = df['timestamp'].max() - df['timestamp'].min()

    all_peaks  = sorted(np.concatenate([left_peaks, right_peaks]))
    peak_times = df['timestamp'].iloc[all_peaks].values

    df['stride_freq_hz'] = np.nan
    for i, row in df.iterrows():
        t = row['timestamp']
        count = np.sum((peak_times >= t - 0.5) & (peak_times <= t + 0.5))
        df.at[i, 'stride_freq_hz'] = count

    # ── Jump height ───────────────────────────────────────────────────────────
    hip_y_series   = df['hip_y'].ffill().bfill()
    baseline_window = max(3, int(FPS))
    hip_baseline   = hip_y_series.rolling(baseline_window, center=True, min_periods=1).median()
    df['jump_height_px'] = (hip_baseline - hip_y_series).clip(lower=0)

    jump_peaks, _ = find_peaks(
        df['jump_height_px'],
        height=10,
        distance=int(FPS * 0.3),
        prominence=8
    )

    # ── Burst / recovery speed ────────────────────────────────────────────────
    BURST_THRESHOLD = df['speed_mps'].quantile(0.75)
    POST_FRAMES     = int(0.5 * FPS)
    burst_frames, _ = find_peaks(
        df['speed_mps'].fillna(0),
        height=BURST_THRESHOLD,
        distance=int(FPS * 0.3)
    )

    df['is_burst'] = False
    df.loc[df.index[burst_frames], 'is_burst'] = True

    # ── Lateral reach ─────────────────────────────────────────────────────────
    df['lat_reach_px']   = abs(df['rwx'] - df['lwx'])
    df['lat_reach_m']    = df['lat_reach_px'] / PX_PER_M_X
    df['hip_cx']         = (df['lhx'].fillna(df['rhx']) + df['rhx'].fillna(df['lhx'])) / 2
    df['left_reach_m']   = abs(df['lwx'] - df['hip_cx']) / PX_PER_M_X
    df['right_reach_m']  = abs(df['rwx'] - df['hip_cx']) / PX_PER_M_X
    df['max_arm_reach_m']= df[['left_reach_m', 'right_reach_m']].max(axis=1)

    # ── Movement Efficiency Index (rolling 2 s window) ────────────────────────
    WINDOW_FR = max(3, int(2.0 * FPS))
    foot_x_np  = df['foot_x_sm'].values
    foot_y_sm_np = savgol_filter(df['foot_y'].ffill().bfill(), WIN, 2)

    mei_values = []
    for i in range(len(df)):
        start    = max(0, i - WINDOW_FR)
        xs       = foot_x_np[start:i+1]
        ys       = foot_y_sm_np[start:i+1]
        dx_mei   = np.diff(xs) / PX_PER_M_X
        dy_mei   = np.diff(ys) / PX_PER_M_Y
        path_len = np.sum(np.sqrt(dx_mei**2 + dy_mei**2))
        net_disp = np.sqrt(((xs[-1] - xs[0]) / PX_PER_M_X)**2 +
                           ((ys[-1] - ys[0]) / PX_PER_M_Y)**2)
        mei = net_disp / path_len if path_len > 0.01 else 1.0
        mei_values.append(min(mei, 1.0))
    df['movement_efficiency'] = mei_values

    # ── Save CSV ──────────────────────────────────────────────────────────────
    output_cols = [
        'frame_id', 'timestamp',
        'speed_mps', 'accel_mps2',
        'court_zone', 'lateral_zone', 'depth_zone',
        'stride_freq_hz',
        'jump_height_px',
        'lat_reach_m', 'max_arm_reach_m',
        'movement_efficiency',
        'is_burst'
    ]
    df[output_cols].to_csv(output_csv_path, index=False)

    # ── Build frame_entries (mirrors notebook cell) ───────────────────────────
    # Build a fast frame_id → raw frame lookup
    raw_frame_lookup = {fr['frame_id']: fr for fr in frames_raw}

    # Smoothed displacement for direction angle
    dx_sm      = df['foot_x_sm'].diff().fillna(0).values
    dy_sm_vals = pd.Series(
        savgol_filter(df['foot_y'].ffill().bfill(), WIN, 2)
    ).diff().fillna(0).values

    frame_entries = []
    for i, row in df.iterrows():
        fid = int(row['frame_id'])

        bb = {
            'x':      round(float(row['bb_x1']), 1),
            'y':      round(float(row['bb_y1']), 1),
            'width':  round(float(row['bb_w']),  1),
            'height': round(float(row['bb_h']),  1),
        }

        src_frame = raw_frame_lookup.get(fid, {})
        raw_kps   = src_frame.get('keypoints', [])
        keypoints = [
            {
                'name':       kp['name'],
                'x':          round(float(kp['x']), 1),
                'y':          round(float(kp['y']), 1),
                'confidence': round(float(kp['confidence']), 3),
            }
            for kp in raw_kps
            if kp.get('confidence', 0) >= CONF_THR
        ]

        court_x = round(float(row['foot_x']) / PX_PER_M_X, 2) \
            if not math.isnan(row['foot_x']) else None
        court_y = round(float(row['foot_y']) / PX_PER_M_Y, 2) \
            if not math.isnan(row['foot_y']) else None

        spd  = round(float(row['speed_mps']),  3) \
            if not math.isnan(row.get('speed_mps',  float('nan'))) else 0.0
        acc  = round(float(row['accel_mps2']), 3) \
            if not math.isnan(row.get('accel_mps2', float('nan'))) else 0.0
        dirn = _movement_direction(dx_sm[i], dy_sm_vals[i])

        lf = {'x': round(float(row['lax']), 1), 'y': round(float(row['lay']), 1)} \
            if not math.isnan(row['lax']) else None
        rf = {'x': round(float(row['rax']), 1), 'y': round(float(row['ray']), 1)} \
            if not math.isnan(row['rax']) else None

        jh   = round(float(row['jump_height_px']), 2)
        step = _classify_step(jh, spd, acc)

        frame_entries.append({
            'frame_id':  fid,
            'timestamp': round(float(row['timestamp']), 4),
            'bounding_box': bb,
            'pose': {'keypoints': keypoints},
            'center_position': {
                'court_x': court_x,
                'court_y': court_y,
            },
            'movement': {
                'speed':        spd,
                'acceleration': acc,
                'direction':    dirn,
            },
            'footwork': {
                'step_type':  step,
                'left_foot':  lf,
                'right_foot': rf,
            },
            'status': {
                'is_moving':    spd > 0.3,
                'is_jumping':   jh > 15,
                'is_recovering': bool(row.get('is_burst', False)),
            },
            'court_zone': str(row['court_zone']),
            'jump_height_px': jh,
        })

    # ── Aggregated metrics (notebook cell logic) ───────────────────────────────
    dx_m_all   = df['foot_x_sm'].diff().fillna(0) / PX_PER_M_X
    dy_m_all   = pd.Series(
        savgol_filter(df['foot_y'].ffill().bfill(), WIN, 2)
    ).diff().fillna(0) / PX_PER_M_Y
    total_distance = float(np.sum(np.sqrt(dx_m_all**2 + dy_m_all**2)))

    avg_speed = float(df['speed_mps'].mean())
    max_speed = float(df['speed_mps'].max())

    # Overall MEI: net A→B displacement / total path length
    net_displacement = float(np.sqrt(
        ((df['foot_x_sm'].iloc[-1] - df['foot_x_sm'].iloc[0]) / PX_PER_M_X)**2 +
        ((df['foot_y'].iloc[-1]    - df['foot_y'].iloc[0])    / PX_PER_M_Y)**2
    ))
    overall_mei = round(min(net_displacement / total_distance, 1.0), 3) \
        if total_distance > 0 else 1.0

    # Court coverage: 10×10 grid, count unique cells
    GRID   = 10
    cx_norm = (df['foot_x'].dropna() / FRAME_W * GRID).astype(int).clip(0, GRID - 1)
    cy_norm = (df['foot_y'].dropna() / FRAME_H * GRID).astype(int).clip(0, GRID - 1)
    cells_visited      = len(set(zip(cx_norm, cy_norm)))
    court_coverage_pct = round(cells_visited / GRID**2 * 100, 1)

    jump_count = int(len(jump_peaks))

    # Average recovery time (40th-percentile speed threshold)
    BURST_END_SPEED = df['speed_mps'].quantile(0.40)
    recovery_times  = []
    speed_vals      = df['speed_mps'].fillna(0).values
    timestamps_arr  = df['timestamp'].values
    for bf in burst_frames:
        for j in range(bf, min(bf + int(FPS * 3), len(speed_vals))):
            if speed_vals[j] < BURST_END_SPEED:
                recovery_times.append(timestamps_arr[j] - timestamps_arr[bf])
                break
    avg_recovery_time = round(float(np.mean(recovery_times)), 3) \
        if recovery_times else 0.0

    # Pose stability: 1 − normalised std of hip_y
    hip_y_std   = df['hip_y'].std()
    hip_y_range = df['hip_y'].max() - df['hip_y'].min()
    pose_stability = round(float(1.0 - min(hip_y_std / (hip_y_range + 1e-6), 1.0)), 3)

    aggregated_metrics = {
        'total_distance_covered':    round(total_distance, 2),
        'average_speed':             round(avg_speed, 3),
        'max_speed':                 round(max_speed, 3),
        'movement_efficiency':       overall_mei,
        'court_coverage_percentage': court_coverage_pct,
        'jump_count':                jump_count,
        'average_recovery_time':     avg_recovery_time,
        'pose_stability_score':      pose_stability,
    }

    # ── Save metrics JSON (includes frame_entries) ────────────────────────────
    output_doc = {
        'match_id':           match_id,
        'player_id':          player_id,
        'frames':             frame_entries,
        'aggregated_metrics': aggregated_metrics,
    }
    with open(output_json_path, 'w') as f:
        json.dump(output_doc, f, indent=2)

    return aggregated_metrics


In [ ]:
%%writefile app/module1/render.py
"""
Module 1 — annotated video rendering (pose overlay + movement overlay).
Content moved verbatim out of the original api-ready notebook's cells 2
(COL / SKELETON_PAIRS), 5 (draw helpers), 7 (render_pose_video), and 9
(render_movement_video).
"""
import math

import cv2
import numpy as np
import json

from .tracking import draw_pose

# ------------------------------------------------------------------
# Colour palette + skeleton pairs (notebook cell 2)
# ------------------------------------------------------------------
COL = {
    'tracked':    (0, 220, 0),
    'predicted':  (0, 165, 255),
    'jump':       (0, 0, 255),
    'sprint':     (255, 100, 0),
    'lunge':      (180, 0, 255),
    'step':       (0, 200, 200),
    'stand':      (180, 180, 180),
    'skeleton':   (0, 255, 255),
    'kp_dot':     (0, 0, 255),
    'text_bg':    (20, 20, 20),
    'text_white': (255, 255, 255),
    'speed_bar':  (0, 200, 100),
    'accel_pos':  (0, 180, 255),
    'accel_neg':  (0, 60, 255),
}

SKELETON_PAIRS = [
    ('left_shoulder', 'left_elbow'),   ('left_elbow', 'left_wrist'),
    ('right_shoulder', 'right_elbow'), ('right_elbow', 'right_wrist'),
    ('left_shoulder', 'right_shoulder'),
    ('left_shoulder', 'left_hip'),     ('right_shoulder', 'right_hip'),
    ('left_hip', 'right_hip'),
    ('left_hip', 'left_knee'),         ('left_knee', 'left_ankle'),
    ('right_hip', 'right_knee'),       ('right_knee', 'right_ankle'),
    ('nose', 'left_eye'),              ('nose', 'right_eye'),
]


# ------------------------------------------------------------------
# Draw helpers (notebook cell 5)
# ------------------------------------------------------------------
def _draw_skeleton(frame, keypoints):
    kp_map = {kp['name']: (int(kp['x']), int(kp['y'])) for kp in keypoints}
    for a, b in SKELETON_PAIRS:
        if a in kp_map and b in kp_map:
            cv2.line(frame, kp_map[a], kp_map[b], COL['skeleton'], 2)
    for pt in kp_map.values():
        cv2.circle(frame, pt, 4, COL['kp_dot'], -1)


def _draw_text_box(frame, lines, origin, font_scale=0.5, thickness=1, padding=5):
    """Dark semi-transparent box then white text."""
    font   = cv2.FONT_HERSHEY_SIMPLEX
    line_h = int(font_scale * 28)
    max_w  = max(cv2.getTextSize(l, font, font_scale, thickness)[0][0] for l in lines)
    x0, y0 = origin
    box_h  = line_h * len(lines) + padding * 2
    box_w  = max_w + padding * 2

    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + box_w, y0 + box_h), COL['text_bg'], -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

    for i, line in enumerate(lines):
        y = y0 + padding + (i + 1) * line_h - 4
        cv2.putText(frame, line, (x0 + padding, y), font, font_scale,
                    COL['text_white'], thickness, cv2.LINE_AA)


def _draw_speed_bar(frame, speed, max_spd=10.0, origin=(20, 80), bar_w=180, bar_h=14):
    x, y = origin
    fill = int(min(speed / max_spd, 1.0) * bar_w)
    cv2.rectangle(frame, (x, y), (x + bar_w, y + bar_h), (60, 60, 60), -1)
    cv2.rectangle(frame, (x, y), (x + fill,  y + bar_h), COL['speed_bar'], -1)
    cv2.rectangle(frame, (x, y), (x + bar_w, y + bar_h), (200, 200, 200), 1)
    cv2.putText(frame, f'{speed:.1f} m/s', (x + bar_w + 6, y + bar_h - 2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, COL['text_white'], 1, cv2.LINE_AA)


def _draw_direction_arrow(frame, cx, cy, direction_deg, speed, length=40):
    if speed < 0.3:
        return
    rad = math.radians(direction_deg)
    ex  = int(cx + length * math.cos(rad))
    ey  = int(cy - length * math.sin(rad))   # image Y inverted
    cv2.arrowedLine(frame, (cx, cy), (ex, ey), (255, 220, 0), 2, tipLength=0.3)


# ------------------------------------------------------------------
# Pose overlay video (notebook cell 7)
# ------------------------------------------------------------------
def render_pose_video(video_path, out_mp4, json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    frames_data = {f['frame_id']: f for f in data['frames']}

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise Exception("Cannot open video")

    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out = cv2.VideoWriter(out_mp4, cv2.VideoWriter_fourcc(*"mp4v"), fps, (frame_w, frame_h))

    for frame_id in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break

        fd = frames_data.get(frame_id)
        if fd and fd.get("player_detected"):
            status = fd.get("tracking_status", "missing")
            box_color = (0, 255, 0) if status == "tracked" else (0, 165, 255)
            bb = fd["bounding_box"]
            if bb is not None:
                x1, y1, x2, y2 = int(bb["x1"]), int(bb["y1"]), int(bb["x2"]), int(bb["y2"])
                cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 3)
                cv2.putText(frame, f"Target Player | {status}", (x1, max(30, y1 - 30)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.65, box_color, 2)
                pose_name = fd.get("shot_classification", "Detecting...")
                shot_conf = fd.get("shot_confidence", 0.0)
                cv2.putText(frame, f"Shot: {pose_name} ({shot_conf:.2f})", (x1, max(55, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)

            # Draw pose
            kpts = np.zeros((17, 2))
            kpt_conf = np.zeros(17)
            for kp in fd.get("keypoints", []):
                idx = kp["id"]
                kpts[idx] = [kp["x"], kp["y"]]
                kpt_conf[idx] = kp["confidence"]
            draw_pose(frame, kpts, kpt_conf)

        out.write(frame)

    cap.release()
    out.release()


# ------------------------------------------------------------------
# Movement overlay video (notebook cell 9)
# ------------------------------------------------------------------
def render_movement_video(json_path, original_video_path, output_video_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    frame_entries = data.get('frames', [])

    if not output_video_path or not original_video_path:
        return

    frame_lookup = {e['frame_id']: e for e in frame_entries}
    jh_lookup = {e['frame_id']: float(e.get('jump_height_px', 0.0)) for e in frame_entries}

    cap = cv2.VideoCapture(original_video_path)
    if not cap.isOpened():
        return

    fw_v    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fh_v    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_out = cap.get(cv2.CAP_PROP_FPS) or 30
    total_fr = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out_vid = cv2.VideoWriter(
        output_video_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps_out, (fw_v, fh_v)
    )

    for video_frame_id in range(total_fr):
        ret, frame = cap.read()
        if not ret:
            break

        entry = frame_lookup.get(video_frame_id)
        if entry is None:
            out_vid.write(frame)
            continue

        bb  = entry.get('bounding_box', {})
        mv  = entry.get('movement', {})
        fw_ = entry.get('footwork', {})
        st  = entry.get('status', {})
        cp  = entry.get('center_position', {})
        pose = entry.get('pose', {})
        kps = pose.get('keypoints', [])

        x1 = int(bb.get('x', 0))
        y1 = int(bb.get('y', 0))
        x2 = int(x1 + bb.get('width', 0))
        y2 = int(y1 + bb.get('height', 0))
        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        step_col = COL.get(fw_.get('step_type', 'stand'), COL['stand'])

        cv2.rectangle(frame, (x1, y1), (x2, y2), step_col, 2)

        if kps:
            _draw_skeleton(frame, kps)

        _draw_direction_arrow(frame, cx, cy, mv.get('direction', 0), mv.get('speed', 0))

        zone_label = f"Zone: {entry.get('court_zone', '?')}"
        flags = []
        if st.get('is_jumping'):    flags.append('JUMP')
        if st.get('is_recovering'): flags.append('RECOVER')
        if not flags and st.get('is_moving'):
            flags.append(str(fw_.get('step_type', '')).upper())
        if not flags:
            flags.append('STAND')

        cp_x = cp.get('court_x') if cp.get('court_x') is not None else '?'
        cp_y = cp.get('court_y') if cp.get('court_y') is not None else '?'

        lines = [
            f"Spd: {mv.get('speed', 0):.1f} m/s  Acc: {mv.get('acceleration', 0):+.1f}",
            f"Dir: {mv.get('direction', 0):.0f}deg  Step: {fw_.get('step_type', '')}",
            f"Court: ({cp_x}, {cp_y}) m",
            zone_label,
            '  '.join(flags),
        ]
        _draw_text_box(frame, lines, (max(0, x1), max(0, y1 - 110)))

        _draw_speed_bar(frame, mv.get('speed', 0), origin=(16, 20))

        for foot_key in ('left_foot', 'right_foot'):
            ft = fw_.get(foot_key)
            if ft:
                cv2.circle(frame, (int(ft['x']), int(ft['y'])), 6, (0, 255, 180), -1)

        jh_px = jh_lookup.get(video_frame_id, 0.0)
        if jh_px > 10:
            jh_int   = int(jh_px)
            jh_bar_x = x2 + 6
            cv2.line(frame, (jh_bar_x, y2), (jh_bar_x, max(y2 - jh_int, 0)), (0, 0, 255), 4)
            cv2.putText(frame, f'J:{jh_int}px', (jh_bar_x + 4, max(y2 - jh_int - 4, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 0, 255), 1, cv2.LINE_AA)

        out_vid.write(frame)

    cap.release()
    out_vid.release()


In [ ]:
%%writefile app/routers/__init__.py


In [ ]:
%%writefile app/routers/module1.py
"""
Module 1 API router — player movement & pose analysis endpoints.

Changes relative to the original api-ready notebook (cells 10-13, 24):

Step 0 (cleanup):
  * Removed the broken shuttle-tracking stub from /api/process-video —
    the `TEMP_DIR = "/kaggle/working"` reassignment inside the function
    body shadowed the three `os.path.join(TEMP_DIR, ...)` calls above it
    (Python treats TEMP_DIR as local to the whole function once it's
    assigned anywhere inside it), so every request raised
    UnboundLocalError before background_tasks.add_task() ever ran.
  * The synchronous, unused `run_shuttle_tracking(...)` call is gone too
    — shuttle tracking gets reintroduced correctly, inside the
    background task, in Step 3.

Step 1 (shared paths + identifiers):
  * TEMP_DIR ("temp_exports") replaced with the shared UPLOADS_DIR /
    OUTPUTS_DIR from app.config, so Module 2's outputs/combine and
    outputs/m2 can live alongside these without collisions.
  * match_id / player_id are now validated with the same
    validate_player_match_metadata() Module 2 already uses, and a
    player_name field was added to match it (previously Module 1 didn't
    collect one). This is a breaking change to the existing request
    contract — the frontend form (Step 15) will need updating to match,
    and any caller still sending free-form ids like "match_001" /
    "player_01" will now get a 400 instead of being silently accepted.
"""
import concurrent.futures
import shutil
import uuid

from fastapi import APIRouter, BackgroundTasks, File, Form, UploadFile
from fastapi.responses import FileResponse, JSONResponse, StreamingResponse

from .. import config
from ..module1.movement import extract_movement_features
from ..module1.render import render_movement_video, render_pose_video
from ..module1.tracking import run_tracking_inference
from ..shared.video_utils import transcode_to_h264

router = APIRouter()

# Background-task progress store, keyed by task_id (notebook cell 10).
# Module 2 will eventually report into this same dict too (Step 13) rather
# than keeping its own separate ANALYSIS_JOBS/ThreadPoolExecutor tracker
# (roadmap finding 0.4) — that unification happens in Step 13, not here.
progress_store = {}


def process_video_task(task_id, video_path, mp4_path, json_path, csv_path, match_id="p001_m0001", player_id="p001"):
    def progress_cb(current, total):
        pct = int((current / total) * 100) if total > 0 else 0
        progress_store[task_id] = {"status": "processing", "progress": pct}

    try:
        metrics = run_tracking_inference(
            video_path=video_path,
            out_mp4=mp4_path,
            out_json=json_path,
            out_csv=csv_path,
            model_path=config.POSE_MODEL_PATH,
            custom_model_path=str(config.MODULE1_CLASSIFIER_MODEL_PATH),
            progress_callback=progress_cb
        )

        # NOTE: these three filenames are still flat (not task_id-prefixed),
        # same as the original notebook — a second request processing
        # concurrently would overwrite these. Pre-existing limitation,
        # unchanged by Steps 0/1; worth revisiting once Step 13 unifies
        # the job model.
        movement_json_path = str(config.OUTPUTS_DIR / "movement_metrics.json")
        movement_csv_path = str(config.OUTPUTS_DIR / "movement_features.csv")
        movement_mp4_path = str(config.OUTPUTS_DIR / "movement_output.mp4")

        movement_metrics = extract_movement_features(
            input_json_path=json_path,
            output_json_path=movement_json_path,
            output_csv_path=movement_csv_path,
            output_video_path=None,
            original_video_path=None,
            match_id=match_id,
            player_id=player_id
        )

        progress_store[task_id]["progress"] = 90

        # Render both videos simultaneously using multi-threading
        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
            f1 = executor.submit(render_pose_video, video_path, mp4_path, json_path)
            f2 = executor.submit(render_movement_video, movement_json_path, video_path, movement_mp4_path)
            concurrent.futures.wait([f1, f2])
            f1.result()
            f2.result()

        # Transcode both rendered videos to browser-playable H.264
        transcode_to_h264(mp4_path, mp4_path.replace(".mp4", "_web.mp4"))
        import os
        os.replace(mp4_path.replace(".mp4", "_web.mp4"), mp4_path)

        transcode_to_h264(movement_mp4_path, movement_mp4_path.replace(".mp4", "_web.mp4"))
        os.replace(movement_mp4_path.replace(".mp4", "_web.mp4"), movement_mp4_path)

        progress_store[task_id]["progress"] = 95

        progress_store[task_id] = {
            "status": "completed",
            "progress": 100,
            "metrics": metrics,
            "movement_metrics": movement_metrics,
            "exports": {
                "json_url": "/api/download/movement_metrics.json",
                "csv_url": "/api/download/output.csv",
                "mp4_url": "/api/download/output.mp4",
                "movement_csv_url": "/api/download/movement_features.csv",
                "movement_json_url": "/api/download/movement_metrics.json",
                "movement_mp4_url": "/api/download/movement_output.mp4"
            }
        }
    except Exception as e:
        import traceback
        traceback.print_exc()  # prints the ACTUAL underlying error to notebook output
        progress_store[task_id] = {"status": "failed", "progress": 0, "error": str(e)}


@router.post("/api/process-video")
async def process_video(
    background_tasks: BackgroundTasks,
    file: UploadFile = File(...),
    match_id: str = Form("p001_m0001"),
    player_id: str = Form("p001"),
    player_name: str = Form("shi_yuqi"),
):
    try:
        player_id, player_name, match_id = config.validate_player_match_metadata(
            player_id, player_name, match_id
        )
    except ValueError as exc:
        return JSONResponse({"error": str(exc)}, status_code=400)

    video_path = config.UPLOADS_DIR / file.filename
    with open(video_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    task_id = str(uuid.uuid4())
    progress_store[task_id] = {"status": "starting", "progress": 0}

    json_path = str(config.OUTPUTS_DIR / "output.json")
    csv_path = str(config.OUTPUTS_DIR / "output.csv")
    mp4_path = str(config.OUTPUTS_DIR / "output.mp4")

    background_tasks.add_task(
        process_video_task, task_id, str(video_path), mp4_path, json_path, csv_path, match_id, player_id
    )

    return {"task_id": task_id}


@router.get("/api/progress/{task_id}")
def get_progress(task_id: str):
    return progress_store.get(task_id, {"status": "not_found", "progress": 0})


@router.get("/api/progress-stream/{task_id}")
async def progress_stream(task_id: str):
    import asyncio
    import json as json_module

    async def event_generator():
        last_progress = -1
        last_status = None
        while True:
            data = progress_store.get(task_id, {"status": "not_found", "progress": 0})
            progress = data.get("progress", 0)
            status = data.get("status")

            if progress != last_progress or status != last_status:
                yield f"data: {json_module.dumps(data)}\n\n"
                last_progress = progress
                last_status = status

            if status in ["completed", "failed", "not_found"]:
                break

            await asyncio.sleep(0.5)

    return StreamingResponse(event_generator(), media_type="text/event-stream")


@router.get("/api/download/{filename}")
async def download_file(filename: str):
    # NOTE: Step 14 will add basename-only validation here
    # (Path(filename).name == filename) to close a path-traversal gap.
    # Left unchanged for now to keep this step's diff scoped to Step 0/1.
    file_path = config.OUTPUTS_DIR / filename
    if file_path.exists():
        media_type = "video/mp4" if filename.endswith(".mp4") else None
        return FileResponse(str(file_path), filename=filename, media_type=media_type)
    return {"error": "File not found"}


In [ ]:
%%writefile app/main.py
"""
FastAPI application entrypoint.

Step 1 reconciles what used to be two separate app.mount("/static", ...)
calls (one implied per notebook) into a single instance here, and adds the
/outputs mount Module 2 will need once it's wired in (Step 13+).
"""
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from fastapi.staticfiles import StaticFiles

from . import config
from .routers import module1

app = FastAPI(title="Badminton Performance Analysis API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

if config.STATIC_DIR.exists():
    app.mount("/static", StaticFiles(directory=str(config.STATIC_DIR)), name="static")

app.mount("/outputs", StaticFiles(directory=str(config.OUTPUTS_DIR)), name="outputs")

app.include_router(module1.router)
# app.include_router(module2.router)  # added starting Step 13


@app.get("/")
async def read_index():
    index_file = config.STATIC_DIR / "index.html"
    if index_file.exists():
        return FileResponse(str(index_file))
    return JSONResponse({"message": "Badminton analysis API is running."})


## Static frontend assets (unchanged from the original notebook — Step 15 will touch these)

In [ ]:
%%writefile static/index.html

<!DOCTYPE html>
<html lang="en">

<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Badminton Player Analytics</title>
  <meta name="description" content="AI-powered badminton pose detection and movement analysis dashboard">
  <link rel="stylesheet" href="/static/style.css">
</head>

<body>
  <div class="app-shell">

    <!-- ── Navbar ─────────────────────────────────────────────────────── -->
    <nav class="navbar">
      <div class="navbar-brand">
        <span class="brand-name">Module 01 - Badminton Pose Detection &amp; Movement Analysis</span>
      </div>
    </nav>

    <!-- ── Main ───────────────────────────────────────────────────────── -->
    <div class="main-content">

      <div id="mainContainer" class="centered-layout">

        <!-- ── LEFT COLUMN ──────────────────────────────────────────── -->
        <div class="left-column">

          <!-- Upload Card -->
          <div class="glass-card" style="margin-bottom: 24px;">
            <div class="card-header">
              <span class="card-title">Upload Video</span>
            </div>

            <form id="uploadForm">
              <label class="upload-area" id="uploadArea">
                <input type="file" id="videoFile" accept="video/mp4,video/mov,video/avi" required>
                <div class="upload-label">Drop your video here</div>
                <div class="upload-hint">MP4 · MOV · AVI — max 2 GB</div>
              </label>

              <div class="selected-file" id="selectedFile" style="display:none;">
                <span class="file-name" id="fileNameDisplay">No file selected</span>
              </div>

              <div class="form-row" style="display: flex; gap: 16px; margin: 20px 24px 0;">
                <div class="form-group" style="flex: 1; display: flex; flex-direction: column; gap: 6px;">
                  <label for="matchId" style="font-size: 0.75rem; font-weight: 600; text-transform: uppercase; color: #6b7280; letter-spacing: 0.05em;">Match ID</label>
                  <input type="text" id="matchId" value="match_001" placeholder="e.g. match_001" style="padding: 10px 14px; border: 1px solid #e5e7eb; border-radius: 8px; font-size: 0.85rem; outline: none; background: #f9fafb; color: #000; transition: border-color 0.2s;" onfocus="this.style.borderColor='#00334F'" onblur="this.style.borderColor='#e5e7eb'">
                </div>
                <div class="form-group" style="flex: 1; display: flex; flex-direction: column; gap: 6px;">
                  <label for="playerId" style="font-size: 0.75rem; font-weight: 600; text-transform: uppercase; color: #6b7280; letter-spacing: 0.05em;">Player ID</label>
                  <input type="text" id="playerId" value="player_01" placeholder="e.g. player_01" style="padding: 10px 14px; border: 1px solid #e5e7eb; border-radius: 8px; font-size: 0.85rem; outline: none; background: #f9fafb; color: #000; transition: border-color 0.2s;" onfocus="this.style.borderColor='#00334F'" onblur="this.style.borderColor='#e5e7eb'">
                </div>
              </div>

              <button type="submit" id="submitBtn" class="btn-primary">
                <span id="submitBtnText">Analyse Video</span>
              </button>
            </form>

            <!-- Progress -->
            <div id="progressSection" class="progress-section" style="display:none;">
              <div class="progress-header">
                <span class="progress-status" id="progressStatus">Processing…</span>
                <span class="progress-pct" id="progressPct">0%</span>
              </div>
              <div class="progress-track">
                <div class="progress-fill" id="progressBar"></div>
              </div>
            </div>

            <div style="padding: 0 24px 20px; display:flex; align-items:center; gap:8px;">
              <div class="status-badge idle" id="statusBadge">
                <div class="status-dot" id="statusDot"></div>
                <span id="statusText">Ready</span>
              </div>
            </div>
          </div>

          <!-- Export Card -->
          <div class="glass-card fade-in" id="exports" style="display:none;">
            <div class="card-header">
              <span class="card-title">Export Data</span>
            </div>
            <div class="export-grid">
              <button id="btnJson" class="export-btn"> JSON</button>
              <button id="btnCsv" class="export-btn"> CSV</button>
              <button id="btnMp4" class="export-btn"> Pose Video</button>
              <button id="btnMovementCsv" class="export-btn">Movement CSV</button>
              <button id="btnMovementMp4" class="export-btn" style="grid-column:span 2;"> Movement Video</button>
            </div>
          </div>

        </div><!-- /left-column -->

        <!-- ── RIGHT COLUMN ─────────────────────────────────────────── -->
        <div class="right-column" style="display:none;">

          <!-- Tracking Summary Card -->
          <div class="glass-card fade-in" id="dashboard" style="display:none;">
            <div class="card-header">
              <span class="card-title">Pose Detection</span>
            </div>
            <div class="tracking-row">
              <div class="tracking-stat">
                <div class="stat-value" id="playerTracked">✓</div>
                <div class="stat-label">Player Locked</div>
              </div>
              <div class="tracking-stat">
                <div class="stat-value" id="keypointsDetected">17</div>
                <div class="stat-label">Keypoints</div>
              </div>
              <div class="tracking-stat">
                <div class="stat-value" id="avgConfidence">—</div>
                <div class="stat-label">Avg Confidence</div>
              </div>
            </div>
          </div>

          <!-- Movement Metrics Card -->
          <div class="glass-card fade-in" id="movementDashboard" style="display:none;">
            <div class="card-header">
              <span class="card-title">Movement Metrics</span>
            </div>
            <div class="metrics-grid">
              <div class="metric-card">
                <div class="metric-label">Distance</div>
                <div class="metric-value" id="totalDist">—</div>
                <div class="metric-unit">metres</div>
              </div>
              <div class="metric-card">
                <div class="metric-label">Avg Speed</div>
                <div class="metric-value" id="avgSpeed">—</div>
                <div class="metric-unit">m/s</div>
              </div>
              <div class="metric-card">
                <div class="metric-label">Max Speed</div>
                <div class="metric-value" id="maxSpeed">—</div>
                <div class="metric-unit">m/s</div>
              </div>
              <div class="metric-card">
                <div class="metric-label">Efficiency</div>
                <div class="metric-value" id="movementEff">—</div>
                <div class="metric-unit">score</div>
              </div>
              <div class="metric-card">
                <div class="metric-label">Coverage</div>
                <div class="metric-value" id="courtCoverage">—</div>
                <div class="metric-unit">% of court</div>
              </div>
              <div class="metric-card">
                <div class="metric-label">Jumps</div>
                <div class="metric-value" id="jumpCount">—</div>
                <div class="metric-unit">detected</div>
              </div>
              <div class="metric-card">
                <div class="metric-label">Recovery</div>
                <div class="metric-value" id="avgRecoveryTime">—</div>
                <div class="metric-unit">sec avg</div>
              </div>
              <div class="metric-card">
                <div class="metric-label">Stability</div>
                <div class="metric-value" id="poseStability">—</div>
                <div class="metric-unit">score</div>
              </div>
            </div>
          </div>

          <!-- Video Player Card -->
          <div class="glass-card fade-in" id="videoPlayerSection" style="display:none;">
            <div class="card-header">
              <span class="card-title">Analysis Output</span>
            </div>

            <div class="dual-video-grid">
              <div class="video-col">
                <div class="video-col-label">
                  Pose Detection
                </div>
                <video id="outputVideo" controls></video>
              </div>
              <div class="video-col">
                <div class="video-col-label">
                  Movement Analysis
                </div>
                <video id="outputMovementVideo" controls></video>
              </div>
            </div>
          </div>

        </div><!-- /right-column -->
      </div><!-- /mainContainer -->
    </div><!-- /main-content -->
  </div><!-- /app-shell -->
  <script src="/static/app.js?v=2.0"></script>
</body>

</html>

In [ ]:
%%writefile static/app.js


const API_ROOT = "";
/* ── Element refs ────────────────────────────────────────────────────────── */
const fileInput       = document.getElementById('videoFile');
const uploadArea      = document.getElementById('uploadArea');
const selectedFile    = document.getElementById('selectedFile');
const fileNameDisplay = document.getElementById('fileNameDisplay');
const submitBtn       = document.getElementById('submitBtn');
const submitBtnText   = document.getElementById('submitBtnText');
const submitBtnIcon   = document.getElementById('submitBtnIcon');

const progressSection = document.getElementById('progressSection');
const progressBar     = document.getElementById('progressBar');
const progressStatus  = document.getElementById('progressStatus');
const progressPct     = document.getElementById('progressPct');

const statusBadge = document.getElementById('statusBadge');
const statusDot   = document.getElementById('statusDot');
const statusText  = document.getElementById('statusText');

/* ── Status helper ───────────────────────────────────────────────────────── */
function setStatus(state, message) {
  statusBadge.className = `status-badge ${state}`;
  statusDot.className   = `status-dot${state === 'working' ? ' pulse' : ''}`;
  statusText.textContent = message;
}

/* ── File drag & drop ────────────────────────────────────────────────────── */
uploadArea.addEventListener('dragover', (e) => {
  e.preventDefault();
  uploadArea.classList.add('drag-over');
});

['dragleave', 'dragend'].forEach(evt =>
  uploadArea.addEventListener(evt, () => uploadArea.classList.remove('drag-over'))
);

uploadArea.addEventListener('drop', (e) => {
  e.preventDefault();
  uploadArea.classList.remove('drag-over');
  const files = e.dataTransfer?.files;
  if (files && files.length > 0) {
    const dt = new DataTransfer();
    dt.items.add(files[0]);
    fileInput.files = dt.files;
    showSelectedFile(files[0].name);
  }
});

fileInput.addEventListener('change', () => {
  if (fileInput.files.length > 0) {
    showSelectedFile(fileInput.files[0].name);
  }
});

function showSelectedFile(name) {
  fileNameDisplay.textContent = name;
  selectedFile.style.display  = 'flex';
  setStatus('idle', 'File selected');
}

/* ── Form submit ─────────────────────────────────────────────────────────── */
document.getElementById('uploadForm').addEventListener('submit', async (e) => {
  e.preventDefault();

  if (fileInput.files.length === 0) {
    setStatus('error', 'Please select a file first');
    return;
  }

  const file     = fileInput.files[0];
  const matchId  = document.getElementById('matchId').value.trim() || 'match_001';
  const playerId = document.getElementById('playerId').value.trim() || 'player_01';

  const formData = new FormData();
  formData.append('file', file);
  formData.append('match_id', matchId);
  formData.append('player_id', playerId);

  // Reset results panels
  ['dashboard', 'movementDashboard', 'exports', 'videoPlayerSection'].forEach(id => {
    document.getElementById(id).style.display = 'none';
  });
  document.getElementById('mainContainer').className = 'centered-layout';
  document.getElementById('mainContainer').querySelector('.right-column').style.display = 'none';

  // UI — uploading state
  submitBtn.disabled      = true;
  submitBtnText.textContent = 'Processing…';
  progressSection.style.display = 'block';
  progressBar.style.width = '0%';
  progressStatus.textContent = 'Uploading…';
  progressPct.textContent = '0%';
  setStatus('working', 'Uploading video…');

  try {
    const response = await fetch(`${API_ROOT}/api/process-video`, { method: 'POST', body: formData });

    if (!response.ok) {
      const err = await response.json();
      throw new Error(err.detail || 'Upload failed');
    }

    const { task_id } = await response.json();
    setStatus('working', 'Inference running…');
    progressStatus.textContent = 'Running pose detection…';

    // ── SSE (Server-Sent Events) ──────────────────────────────────────────
    const eventSource = new EventSource(`${API_ROOT}/api/progress-stream/${task_id}`);

    eventSource.onmessage = (event) => {
        const data = JSON.parse(event.data);

        if (data.status === 'processing' || data.status === 'starting') {
          const pct = data.progress ?? 0;
          progressBar.style.width   = `${pct}%`;
          progressPct.textContent   = `${pct}%`;
          if (pct < 50) {
            progressStatus.textContent = 'Detecting poses…';
          } else if (pct < 90) {
            progressStatus.textContent = 'Analysing movement…';
          } else {
            progressStatus.textContent = 'Rendering videos simultaneously...';
          }

        } else if (data.status === 'completed') {
          eventSource.close();
          progressBar.style.width   = '100%';
          progressPct.textContent   = '100%';
          progressStatus.textContent = 'Complete!';
          setStatus('success', 'Analysis complete');

          submitBtn.disabled        = false;
          submitBtnText.textContent = 'Analyse Again';

          // Switch to two-column
          document.getElementById('mainContainer').className = 'two-column-layout';
          const rightCol = document.getElementById('mainContainer').querySelector('.right-column');
          rightCol.style.display = 'flex';

          // ── Pose detection section ───────────────────────────────────
          const dash = document.getElementById('dashboard');
          dash.style.display = 'block';
          if (data.metrics) {
            document.getElementById('avgConfidence').textContent =
              data.metrics.average_confidence != null
                ? `${data.metrics.average_confidence}%`
                : '—';
          }

          // ── Videos ──────────────────────────────────────────────────
          const videoSection = document.getElementById('videoPlayerSection');
          videoSection.style.display = 'block';

          const poseVid = document.getElementById('outputVideo');
          poseVid.src   = (data.exports.mp4_url) + '?t=' + Date.now();
          poseVid.load();

          const mvVid = document.getElementById('outputMovementVideo');
          mvVid.src   = (data.exports.movement_mp4_url) + '?t=' + Date.now();
          mvVid.load();



          // ── Movement metrics ─────────────────────────────────────────
          const mvDash = document.getElementById('movementDashboard');
          mvDash.style.display = 'block';
          const mm = data.movement_metrics;
          if (mm) {
            document.getElementById('totalDist').textContent      = mm.total_distance_covered ?? '—';
            document.getElementById('avgSpeed').textContent       = mm.average_speed ?? '—';
            document.getElementById('maxSpeed').textContent       = mm.max_speed ?? '—';
            document.getElementById('movementEff').textContent    = mm.movement_efficiency ?? '—';
            document.getElementById('courtCoverage').textContent  =
              mm.court_coverage_percentage != null ? `${mm.court_coverage_percentage}` : '—';
            document.getElementById('jumpCount').textContent       = mm.jump_count ?? '—';
            document.getElementById('avgRecoveryTime').textContent = mm.average_recovery_time ?? '—';
            document.getElementById('poseStability').textContent   = mm.pose_stability_score ?? '—';
          }

          // ── Exports ──────────────────────────────────────────────────
          const exDiv = document.getElementById('exports');
          exDiv.style.display = 'block';
          document.getElementById('btnJson').onclick        = () => window.location.href = data.exports.json_url;
          document.getElementById('btnCsv').onclick         = () => window.location.href = data.exports.csv_url;
          document.getElementById('btnMp4').onclick         = () => window.location.href = data.exports.mp4_url;
          document.getElementById('btnMovementCsv').onclick = () => window.location.href = data.exports.movement_csv_url;
          document.getElementById('btnMovementMp4').onclick = () => window.location.href = data.exports.movement_mp4_url;

        } else if (data.status === 'failed') {
          eventSource.close();
          setStatus('error', `Error: ${data.error || 'Processing failed'}`);
          progressStatus.textContent = 'Processing failed';
          submitBtn.disabled      = false;
          submitBtnText.textContent = 'Retry';
        }
    };

    eventSource.onerror = (err) => {
      eventSource.close();
      setStatus('error', 'Lost connection to server');
    };

  } catch (err) {
    setStatus('error', err.message);
    submitBtn.disabled        = false;
    submitBtnText.textContent = 'Retry';
  }
});

In [ ]:
%%writefile static/style.css

/* ── Tailwind CSS ────────────────────────────────────────────────────────── */
@import url('https://cdn.tailwindcss.com');
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&display=swap');

/* ── Tailwind Configuration ──────────────────────────────────────────────── */
@layer base {
  :root {
    --color-primary: #00334F;
    --color-white: #ffffff;
    --color-black: #000000;
    --color-gray-50: #f9fafb;
    --color-gray-100: #f3f4f6;
    --color-gray-200: #e5e7eb;
    --color-gray-300: #d1d5db;
    --color-gray-400: #9ca3af;
    --color-gray-500: #6b7280;
    --color-gray-600: #4b5563;
    --color-gray-700: #374151;
    --color-gray-800: #1f2937;
    --color-gray-900: #111827;

    --transition: 0.25s cubic-bezier(0.4,0,0.2,1);
  }

  * {
    @apply transition-colors duration-[var(--transition)];
  }

  body {
    @apply font-sans antialiased;
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
  }
}

@layer components {
  .glass-card {
    @apply bg-white border border-gray-200 rounded-2xl shadow-md;
  }

  .glass-card:hover {
    @apply border-gray-300 shadow-lg;
  }

  .btn-primary {
    @apply px-6 py-3 bg-[#00334F] text-white font-bold rounded-lg transition-all duration-[var(--transition)] hover:bg-gray-800 hover:shadow-lg active:translate-y-0 disabled:opacity-55 disabled:cursor-not-allowed;
  }

  .status-badge {
    @apply inline-flex items-center gap-1.5 px-3 py-1 text-xs font-semibold rounded-full;
  }

  .status-badge.idle {
    @apply bg-gray-100 text-gray-600;
  }

  .status-badge.working {
    @apply bg-blue-100 text-[#00334F];
  }

  .status-badge.success {
    @apply bg-green-100 text-green-700;
  }

  .status-badge.error {
    @apply bg-red-100 text-red-700;
  }

  .export-btn {
    @apply px-4 py-2 bg-gray-50 border border-gray-200 text-gray-700 rounded-lg font-semibold text-xs transition-all duration-[var(--transition)] hover:bg-gray-100 hover:border-gray-300 hover:text-[#00334F] hover:shadow-md;
  }
}


/* ── Reset & Base ────────────────────────────────────────────────────────── */
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

html { scroll-behavior: smooth; }

body {
  font-family: 'Inter', sans-serif;
  background-color: #ffffff;
  color: #000000;
  min-height: 100vh;
  overflow-x: hidden;
}

/* ── Scrollbar ───────────────────────────────────────────────────────────── */
::-webkit-scrollbar { width: 8px; }
::-webkit-scrollbar-track { background: transparent; }
::-webkit-scrollbar-thumb { background: #d1d5db; border-radius: 99px; }
::-webkit-scrollbar-thumb:hover { background: #9ca3af; }

/* ── App Shell ───────────────────────────────────────────────────────────── */
.app-shell {
  display: flex;
  flex-direction: column;
  min-height: 100vh;
}

/* ── Navigation Bar ─────────────────────────────────────────────────────── */
.navbar {
  display: flex;
  align-items: center;
  justify-content: space-between;
  padding: 0 40px;
  height: 64px;
  background: #ffffff;
  border-bottom: 1px solid #e5e7eb;
  position: sticky;
  top: 0;
  z-index: 100;
}

.navbar-brand {
  display: flex;
  align-items: center;
  gap: 12px;
}

.brand-icon {
  width: 36px;
  height: 36px;
  background: #00334F;
  border-radius: 10px;
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 18px;
  flex-shrink: 0;
  color: white;
}

.brand-name {
  font-size: 1.05rem;
  font-weight: 700;
  letter-spacing: -0.02em;
  color: #00334F;
}

.navbar-badge {
  font-size: 0.68rem;
  font-weight: 600;
  letter-spacing: 0.1em;
  text-transform: uppercase;
  color: #6b7280;
  background: #f3f4f6;
  border: 1px solid #e5e7eb;
  border-radius: 999px;
  padding: 3px 12px;
}

/* ── Main Content ────────────────────────────────────────────────────────── */
.main-content {
  flex: 1;
  padding: 40px 36px;
  width: 100%;
  background-color: #ffffff;
}

/* ── Page Title ──────────────────────────────────────────────────────────── */
.page-header {
  margin-bottom: 36px;
}

.page-title {
  font-size: 2rem;
  font-weight: 800;
  letter-spacing: -0.04em;
  line-height: 1.1;
  color: #00334F;
}

.page-subtitle {
  margin-top: 8px;
  font-size: 0.92rem;
  color: #6b7280;
  font-weight: 400;
}

/* ── Glass Card (Light Theme) ───────────────────────────────────────────── */
.glass-card {
  background: #ffffff;
  border: 1px solid #e5e7eb;
  border-radius: 24px;
  box-shadow: 0 2px 8px rgba(0,0,0,0.05);
  transition: border-color 0.25s cubic-bezier(0.4,0,0.2,1), box-shadow 0.25s cubic-bezier(0.4,0,0.2,1);
}

.glass-card:hover {
  border-color: #d1d5db;
  box-shadow: 0 4px 16px rgba(0,0,0,0.08);
}

.card-header {
  display: flex;
  align-items: center;
  gap: 10px;
  padding: 22px 24px 0;
  margin-bottom: 20px;
}

.card-icon {
  width: 32px;
  height: 32px;
  border-radius: 8px;
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 15px;
  flex-shrink: 0;
  background: #f3f4f6;
}

.card-title {
  font-size: 0.9rem;
  font-weight: 600;
  letter-spacing: 0.04em;
  margin-right: 40px;
  margin-left: 40px;
  text-transform: uppercase;
  color: #6b7280;
}

/* ── Upload Area ────────────────────────────────────────────────────────── */
.upload-area {
  border: 2px dotted #9ca3af;
  border-radius: 16px;
  padding: 42px 24px;
  text-align: center;
  cursor: pointer;
  transition: all 0.25s cubic-bezier(0.4,0,0.2,1);
  background: #f9fafb;
  position: relative;
  margin: 0 24px;
  margin-bottom: 60px;
  display: block;
}

.upload-area:hover {
  border-color: #9ca3af;
  background: #f3f4f6;
}

.upload-area.drag-over {
  border-color: #00334F;
  background: #f0f6ff;
}

.upload-label {
  font-size: 1rem;
  font-weight: 600;
  color: #000000;
  margin-bottom: 6px;
  pointer-events: none;

}

.upload-hint {
  font-size: 0.8rem;
  color: #6b7280;
  pointer-events: none;
}

.upload-area input[type="file"] { display: none; }

.selected-file {
  display: flex;
  align-items: center;
  gap: 10px;
  background: #f0f6ff;
  border: 1px solid #d1d5db;
  border-radius: 10px;
  padding: 10px 16px;
  margin: 14px 24px 0;
}

.selected-file .file-icon { font-size: 1.2rem; }
.selected-file .file-name {
  font-size: 0.85rem;
  font-weight: 500;
  color: #00334F;
  white-space: nowrap;
  overflow: hidden;
  text-overflow: ellipsis;
}

/* ── Primary Button ──────────────────────────────────────────────────────── */
.btn-primary {
  display: flex;
  align-items: center;
  justify-content: center;
  gap: 8px;
  width: calc(100% - 48px);
  margin: 18px 24px 24px;
  padding: 14px;
  background: #00334F;
  color: #fff;
  font-family: inherit;
  font-size: 0.95rem;
  font-weight: 700;
  letter-spacing: 0.02em;
  border: none;
  border-radius: 10px;
  cursor: pointer;
  transition: all 0.25s cubic-bezier(0.4,0,0.2,1);
  position: relative;
  overflow: hidden;
  box-shadow: 0 2px 8px rgba(0,51,79,0.15);
}

.btn-primary:hover:not(:disabled) {
  transform: translateY(-2px);
  box-shadow: 0 4px 16px rgba(0,51,79,0.25);
  background: #002340;
}

.btn-primary:active:not(:disabled) { transform: translateY(0); }

.btn-primary:disabled {
  opacity: 0.55;
  cursor: not-allowed;
  box-shadow: none;
}

/* ── Progress ────────────────────────────────────────────────────────────── */
.progress-section {
  padding: 0 24px 24px;
}

.progress-header {
  display: flex;
  justify-content: space-between;
  align-items: center;
  margin-bottom: 10px;
}

.progress-status {
  font-size: 0.82rem;
  font-weight: 500;
  color: #6b7280;
}

.progress-pct {
  font-size: 0.82rem;
  font-weight: 700;
  color: #00334F;
}

.progress-track {
  width: 100%;
  height: 6px;
  background: #e5e7eb;
  border-radius: 99px;
  overflow: hidden;
}

.progress-fill {
  height: 100%;
  width: 0%;
  background: #00334F;
  border-radius: 99px;
  transition: width 0.35s ease;
}

/* ── Status Badge ────────────────────────────────────────────────────────── */
    
.status-badge {
  display: inline-flex;
  align-items: center;
  gap: 6px;
  font-size: 0.78rem;
  font-weight: 600;
  border-radius: 999px;
  padding: 4px 12px;
  margin-top: 12px;
}

.status-badge.idle     { background: #f3f4f6; color: #6b7280; }
.status-badge.working  { background: #f0f6ff; color: #00334F; }
.status-badge.success  { background: #dcfce7; color: #166534; }
.status-badge.error    { background: #fee2e2; color: #b91c1c; }

.status-dot {
  width: 7px; height: 7px;
  border-radius: 50%;
  background: currentColor;
}

.status-dot.pulse {
  animation: pulse-dot 1.4s ease-in-out infinite;
}

@keyframes pulse-dot {
  0%,100% { opacity: 1; transform: scale(1); }
  50%     { opacity: 0.4; transform: scale(0.7); }
}

/* ── Layout Grid ─────────────────────────────────────────────────────────── */
#mainContainer {
  display: grid;
  gap: 28px;
}

#mainContainer.centered-layout {
  grid-template-columns: minmax(0, 640px);
  justify-content: center;
}

#mainContainer.two-column-layout {
  grid-template-columns: 420px 1fr;
  align-items: start;
}

#mainContainer.two-column-layout .right-column {
  display: flex;
  flex-direction: column;
  gap: 24px;
}

@media (max-width: 768px) {
  #mainContainer.two-column-layout {
    grid-template-columns: 1fr;
  }
  .main-content { padding: 24px 16px; }
  .navbar { padding: 0 16px; }
}

/* ── Metric Grid ─────────────────────────────────────────────────────────── */
.metrics-grid {
  display: grid;
  grid-template-columns: repeat(auto-fill, minmax(140px, 1fr));
  gap: 14px;
  padding: 0 24px 24px;
}

.metric-card {
  background: #f9fafb;
  border: 1px solid #e5e7eb;
  border-radius: 10px;
  padding: 16px;
  text-align: center;
  transition: all 0.25s cubic-bezier(0.4,0,0.2,1);
  position: relative;
  overflow: hidden;
}

.metric-card:hover {
  border-color: #d1d5db;
  box-shadow: 0 2px 8px rgba(0,0,0,0.05);
}

.metric-label {
  font-size: 0.7rem;
  font-weight: 600;
  letter-spacing: 0.06em;
  text-transform: uppercase;
  color: #6b7280;
  margin-bottom: 10px;
}

.metric-value {
  font-size: 1.55rem;
  font-weight: 800;
  letter-spacing: -0.03em;
  line-height: 1;
  color: #00334F;
}

.metric-unit {
  font-size: 0.7rem;
  color: #6b7280;
  margin-top: 4px;
  font-weight: 500;
}

/* ── Video Player ────────────────────────────────────────────────────────── */
.dual-video-grid {
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 24px;
  padding: 0 24px 24px;
}

.video-col {
  display: flex;
  flex-direction: column;
  gap: 12px;
}

.video-col-label {
  font-size: 0.85rem;
  font-weight: 600;
  color: #6b7280;
  display: flex;
  align-items: center;
  gap: 8px;
  padding-left: 4px;
}

video {
  width: 100%;
  height: 500px;
  object-fit: contain;
  border-radius: 16px;
  background: #000;
  display: block;
  box-shadow: 0 4px 16px rgba(0,0,0,0.1);
  border: 1px solid #e5e7eb;
}

/* ── Export Buttons ──────────────────────────────────────────────────────── */
.export-grid {
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 10px;
  padding: 0 24px 24px;
}

.export-btn {
  display: flex;
  align-items: center;
  justify-content: center;
  gap: 7px;
  padding: 11px 10px;
  background: #f9fafb;
  border: 1px solid #e5e7eb;
  border-radius: 10px;
  color: #6b7280;
  font-family: inherit;
  font-size: 0.78rem;
  font-weight: 600;
  cursor: pointer;
  transition: all 0.25s cubic-bezier(0.4,0,0.2,1);
  text-decoration: none;
}

.export-btn:hover {
  background: #f0f6ff;
  border-color: #d1d5db;
  color: #00334F;
  transform: translateY(-2px);
  box-shadow: 0 2px 8px rgba(0,51,79,0.1);
}

.export-btn .btn-icon { font-size: 1rem; }

/* ── Tracking Summary Row ────────────────────────────────────────────────── */
.tracking-row {
  display: grid;
  grid-template-columns: repeat(3, 1fr);
  gap: 14px;
  padding: 0 24px 24px;
}

.tracking-stat {
  background: #f9fafb;
  border: 1px solid #e5e7eb;
  border-radius: 10px;
  padding: 16px;
  text-align: center;
}

.tracking-stat .stat-value {
  font-size: 1.6rem;
  font-weight: 800;
  letter-spacing: -0.04em;
  color: #00334F;
}

.tracking-stat .stat-label {
  font-size: 0.68rem;
  font-weight: 600;
  letter-spacing: 0.06em;
  text-transform: uppercase;
  color: #6b7280;
  margin-top: 4px;
}

/* ── Section divider ──────────────────────────────────────────────────────── */
.section-divider {
  height: 1px;
  background: #e5e7eb;
  margin: 0 24px 20px;
}

/* ── Fade-in animation ───────────────────────────────────────────────────── */
@keyframes fadeUp {
  from { opacity: 0; transform: translateY(16px); }
  to   { opacity: 1; transform: translateY(0); }
}

.fade-in {
  animation: fadeUp 0.45s cubic-bezier(0.22, 1, 0.36, 1) forwards;
}

/* ── Footer ──────────────────────────────────────────────────────────────── */
.app-footer {
  text-align: center;
  padding: 20px;
  font-size: 0.72rem;
  color: #6b7280;
  border-top: 1px solid #e5e7eb;
  margin-top: 40px;
  background-color: #ffffff;
}


In [ ]:
import sys
sys.path.insert(0, str(BASE))

# Reload from disk in case this cell is re-run after editing the writefile cells above.
for mod in list(sys.modules):
    if mod == "app" or mod.startswith("app."):
        del sys.modules[mod]

from app.main import app
print("App loaded OK:", app.title)

## Step 0 regression test

Confirms the exact failure mode from finding 0.1 is gone: the endpoint must
return `{"task_id": ...}` immediately, and a bad video must fail through
`progress_store` (status `"failed"`) rather than crashing the request.
Swap in a real, short test clip to also exercise the full pipeline.

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

# 1. Bad match_id/player_id -> should be rejected with 400, not crash
r = client.post(
    "/api/process-video",
    data={"match_id": "match_001", "player_id": "player_01", "player_name": "Test"},
    files={"file": ("test.mp4", b"placeholder bytes", "video/mp4")},
)
print("Invalid ids ->", r.status_code, r.json())

# 2. Valid ids with a REAL short video file -> replace TEST_VIDEO_PATH
TEST_VIDEO_PATH = None  # e.g. "/kaggle/input/your-dataset/sample_rally.mp4"

if TEST_VIDEO_PATH:
    with open(TEST_VIDEO_PATH, "rb") as f:
        r = client.post(
            "/api/process-video",
            data={"match_id": "p001_m0001", "player_id": "p001", "player_name": "Test Player"},
            files={"file": (os.path.basename(TEST_VIDEO_PATH), f, "video/mp4")},
        )
    print("Valid ids ->", r.status_code, r.json())
    task_id = r.json()["task_id"]
    print("Poll /api/progress/{task_id} or /api/progress-stream/{task_id} to watch it complete.")
else:
    print("Set TEST_VIDEO_PATH to a real short clip to exercise the full pipeline end to end.")

## Launch the server

**Security note:** the original notebook had a real ngrok auth token
hardcoded in this cell. That token is now sitting in a file you shared —
treat it as compromised and roll it from your ngrok dashboard. This version
reads the token from an environment variable instead. In Kaggle, add it
under *Add-ons → Secrets* as `NGROK_AUTH_TOKEN`, or set it directly below
for a quick local test (just don't commit/share the notebook afterwards).

In [ ]:
import threading
import nest_asyncio
import uvicorn
from pyngrok import ngrok

NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN")  # set via Kaggle Secrets
if not NGROK_AUTH_TOKEN:
    raise RuntimeError(
        "Set the NGROK_AUTH_TOKEN environment variable (Kaggle Secrets) before running this cell."
    )

nest_asyncio.apply()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

def run_server():
    server.run()

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

## Next up

Step 2 (TrackNetV3 environment setup) and Step 3 (shuttle detection wired
into this same background task) are next, per the roadmap — say the word
when you want to move on to those.